### DATA ingestion

In [2]:
###Document Structure 

from langchain_core.documents import Document

In [6]:
doc=Document(
    page_content="Hello, world!",
    metadata={
        "source": "example.pdf",
        "page": 1,
        "author": "Hirak Doe",
        "date-created": "2021-01-01"
    }
)

doc



Document(metadata={'source': 'example.pdf', 'page': 1, 'author': 'Hirak Doe', 'date-created': '2021-01-01'}, page_content='Hello, world!')

In [10]:
## create a small text file 

import os 
os.makedirs("../data/text_files", exist_ok=True)

In [21]:
sample_texts = {
    "../data/text_files/python_intro2.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support
- good speed
"""
}

for path, content in sample_texts.items():
    # print(content)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

print(f"Created {len(sample_texts)} text file(s)")

Created 1 text file(s)


In [35]:
### TEXT Loader 
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro2.txt", encoding="utf-8")

document = loader.load()

print(document)

[Document(metadata={'source': '../data/text_files/python_intro2.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n- good speed\n')]


In [ ]:
###Directry loader 

from langchain_community.document_loaders import DirectoryLoader

## load all the text from the directory 

dir_loader = DirectoryLoader (
    "../data/text_files",
    glob="**/*.txt",
    loader_cls= TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False,
    )

documents = dir_loader.load()

print(documents)



2


In [56]:
from langchain_community.document_loaders import PyMuPDFLoader, PyMuPDFLoader

## load all the text from the directory 

dir_loader = DirectoryLoader (
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,  
    # loader_kwargs={'encoding': 'utf-8'},
    show_progress=False,
    )

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m155 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/SAFI_SOW-04_Root_Cause_Analyzer_OpenAPI_Harness.pdf', 'file_path': '../data/pdf/SAFI_SOW-04_Root_Cause_Analyzer_OpenAPI_Harness.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'SAFI_SOW-04_Root_Cause_Analyzer_OpenAPI_Harness (1)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content="SAFI | SOW-04 | ROOT CAUSE ANALYZER \nSAFI \nSOW-04\u200b\nRoot Cause Analyzer for OpenAPI API \nTesting Harness \nSix-Hour Product SOW | Distribution Copy \nCENTRAL ACCEPTANCE PRINCIPLE \nEach developer is accepted by cross-testing the other two developers' API Gateways. Self-testing is useful during \ndevelopment but does not count as final acceptance. All three Gateways, Swagger UI documentation, the automated \nHarness and the Root Cause Analyzer must share one protected canonical OpenAPI 3

Embedding and vectorStoreDB

In [47]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model 
import chromadb
from chromadb. config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/hirak/MEGA/rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [52]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print (f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print (f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()



## inisilize the embedding manager 

embedding_manager = EmbeddingManager()
embedding_manager

## generate embeddings for a list of texts 

# texts = ["Hello, world!", "This is a test", "Embedding and vectorStoreDB"]

    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9938.19it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/h1/840gqx7x7ldcq8b44nytvb5c0000gn/T/ipykernel_9690/1439547467.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vector Store 

In [54]:
from typing import Any


class VectorStore:
    """"Manage documents embedings in a chromaDB vector store """
    def __init__(self, collection_name:str="pdf_documents", persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection 
            persist_directory: Directory to save the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection""" 
        try:
            # Create persistent ChromaDB client
            os.makedirs (self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client. get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print (f"Vector store initialized. Collection: {self. collection_name}")
            print(f"Existing documents in collection: {self.collection.count ()}")
        except Exception as e:
            print (f"Error initializing vector store: {e}")
            raise
        


    def add_documents (self, documents: List[Any], embeddings: np.ndarray) :
        """Add documents and their embeddings to the vector store
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len (documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
        # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append (doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc. page_content)
            metadatas.append(metadata)

            # Prepare documents text
            documents_text.append(doc.page_content)

            # Prepare embeddings list
            embeddings_list.append(embedding.tolist())
        
        #Add to collection 
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"totale dcumnet Collection size: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore
        
        
        



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
chunk

NameError: name 'chunks' is not defined